# GMFlow Posterior Re-derivation: Schedule-Agnostic Form

Re-derives `gmflow_posterior_mean_jit` keeping `(alpha_t, sigma_t)` as free parameters.  
Current code hardcodes `alpha = 1 - sigma` (linear/VE). Goal: work for any noise schedule,  
including trig/VP (`alpha = cos(πt/2)`, `sigma = sin(πt/2)`).

**Source:** `repos/piFlow/lakonlab/models/diffusions/gmflow.py`, lines 44–72

## Section 1 — Symbol Setup

In [ ]:
from sympy import *

# Schedule parameters — free positive reals, NO relationship imposed between them
alpha_t, sigma_t = symbols('alpha_t sigma_t', positive=True)
alpha_s, sigma_s = symbols('alpha_s sigma_s', positive=True)

# Observation variables
x_t, x_s = symbols('x_t x_s', real=True)

# GM component parameters (scalar stand-ins; channel sum handled separately)
mu_k, var_k = symbols('mu_k var_k', real=True, positive=True)

# Confirm no schedule relationship
print('Symbols defined. No alpha = f(sigma) relationship imposed.')
print('alpha_t, sigma_t:', alpha_t, sigma_t)
print('alpha_s, sigma_s:', alpha_s, sigma_s)

## Section 2 — Likelihood in Information Form

Forward process: $x_t = \alpha_t x_0 + \sigma_t \varepsilon$, so

$$p(x_t \mid x_0) = \mathcal{N}(\alpha_t x_0,\; \sigma_t^2)$$

In **information (canonical) form** $\mathcal{N}^{-1}(\eta, \Lambda)$ where $\Lambda$ is precision and $\eta$ is the information vector:

$$\Lambda_t = \frac{\alpha_t^2}{\sigma_t^2}, \qquad \eta_t = \frac{\alpha_t}{\sigma_t^2} x_t$$

In [ ]:
# Precision (scalar) and information vector for the t-observation
Lambda_t = alpha_t**2 / sigma_t**2
eta_t    = alpha_t * x_t / sigma_t**2

# Same for the source observation s
Lambda_s = alpha_s**2 / sigma_s**2
eta_s    = alpha_s * x_s / sigma_s**2

print('Precision (t):', Lambda_t)
print('Info vector (t):', eta_t)
print()
print('Precision (s):', Lambda_s)
print('Info vector (s):', eta_s)

## Section 3 — Posterior Update (core derivation)

**Setup:**  
- GM prior from source observation: $p(x_0 \mid x_s) = \sum_k w_k \mathcal{N}(\mu_k, \mathrm{var}_k)$
- New observation: $p(x_t \mid x_0) = \mathcal{N}(\alpha_t x_0, \sigma_t^2)$
- Goal: compute $p(x_0 \mid x_s, x_t)$

In information form, multiplying two Gaussians adds precisions and information vectors.  
The **gain** from adding the $t$-observation on top of the $s$-prior:

$$\zeta = \Lambda_t - \Lambda_s = \left(\frac{\alpha_t}{\sigma_t}\right)^2 - \left(\frac{\alpha_s}{\sigma_s}\right)^2$$

$$\nu = \eta_t - \eta_s = \frac{\alpha_t x_t}{\sigma_t^2} - \frac{\alpha_s x_s}{\sigma_s^2}$$

In [ ]:
# Precision gain and information vector gain
zeta = Lambda_t - Lambda_s
nu   = eta_t   - eta_s

zeta_simplified = simplify(zeta)
nu_simplified   = simplify(nu)

print('zeta =', zeta_simplified)
print('nu   =', nu_simplified)

### Per-component posterior

For component $k$ with prior $\mathcal{N}(\mu_k, \mathrm{var}_k)$:

- Updated precision: $\Lambda_k^{\text{post}} = 1/\mathrm{var}_k + \zeta$
- Updated variance: $\mathrm{var}_k^{\text{post}} = 1 / \Lambda_k^{\text{post}} = \mathrm{var}_k / (\mathrm{var}_k \zeta + 1)$
- Denominator: $d_k = \mathrm{var}_k \zeta + 1$
- Updated mean: $\mu_k^{\text{post}} = \mathrm{var}_k^{\text{post}} \cdot (\eta_k^{\text{prior}} + \nu) = \dfrac{\mathrm{var}_k \nu + \mu_k}{d_k}$
- Log-weight delta: $\Delta \log w_k = \dfrac{\mu_k^T (\nu - \tfrac{1}{2}\zeta\,\mu_k)}{d_k}$ (from completing the square)

In [ ]:
# Denominator
denom_k = var_k * zeta + 1

# Posterior mean for component k
out_mean_k = (var_k * nu + mu_k) / denom_k

# Log-weight delta (scalar; in practice summed over channel dim)
logweight_delta_k = mu_k * (nu - Rational(1,2) * zeta * mu_k) / denom_k

print('denom_k           =', simplify(denom_k))
print('out_mean_k        =', simplify(out_mean_k))
print('logweight_delta_k =', simplify(logweight_delta_k))

### Display as LaTeX

In [ ]:
print('zeta:              ', latex(zeta_simplified))
print('nu:                ', latex(nu_simplified))
print('denom_k:           ', latex(simplify(denom_k)))
print('out_mean_k:        ', latex(simplify(out_mean_k)))
print('logweight_delta_k: ', latex(simplify(logweight_delta_k)))

## Section 4 — Verification: Linear Schedule Reduces to Existing Code

Substitute `alpha = 1 - sigma` and confirm match with `gmflow_posterior_mean_jit` (lines 54–67).

In [ ]:
# Substitute linear schedule into general formulas
linear_subs = {
    alpha_t: 1 - sigma_t,
    alpha_s: 1 - sigma_s,
}

zeta_lin = simplify(zeta.subs(linear_subs))
nu_lin   = simplify(nu.subs(linear_subs))
denom_lin = simplify(denom_k.subs(linear_subs))
mean_lin  = simplify(out_mean_k.subs(linear_subs))

print('=== Linear schedule ===')
print('zeta  =', zeta_lin)
print('nu    =', nu_lin)
print('denom =', denom_lin)
print('mean  =', mean_lin)
print()

# Cross-check: existing code computes
#   alpha_over_sigma_t = (1 - sigma_t) / sigma_t
#   zeta = alpha_over_sigma_t**2 - alpha_over_sigma_s**2
#   nu   = alpha_over_sigma_t * x_t / sigma_t - alpha_over_sigma_s * x_s / sigma_s
alpha_over_sigma_t = (1 - sigma_t) / sigma_t
alpha_over_sigma_s = (1 - sigma_s) / sigma_s
zeta_code = alpha_over_sigma_t**2 - alpha_over_sigma_s**2
nu_code   = alpha_over_sigma_t * x_t / sigma_t - alpha_over_sigma_s * x_s / sigma_s

diff_zeta = simplify(zeta_lin - zeta_code)
diff_nu   = simplify(nu_lin   - nu_code)

print('zeta match (should be 0):', diff_zeta)
print('nu   match (should be 0):', diff_nu)

## Section 5 — Verification: Trig Schedule (VP / TrigFlow)

Substitute `alpha = cos(πt/2)`, `sigma = sin(πt/2)`. Check limits.

In [ ]:
t, s = symbols('t s', positive=True)

trig_subs = {
    alpha_t: cos(pi*t/2),
    sigma_t: sin(pi*t/2),
    alpha_s: cos(pi*s/2),
    sigma_s: sin(pi*s/2),
}

zeta_trig = trigsimp(zeta.subs(trig_subs))
nu_trig   = trigsimp(nu.subs(trig_subs))

print('=== Trig (VP) schedule ===')
print('zeta =', zeta_trig)
print('nu   =', nu_trig)
print()

# Check: VP identity alpha^2 + sigma^2 = 1
vp_identity = simplify(cos(pi*t/2)**2 + sin(pi*t/2)**2 - 1)
print('VP identity (cos²+sin²-1, should be 0):', vp_identity)

In [ ]:
# Boundary limits of out_mean_k under trig schedule
# t -> 0: observation x_t becomes noiseless (sigma_t -> 0), posterior should -> x_t
# t -> 1: sigma_t -> 1, alpha_t -> 0, information -> 0, posterior -> prior mean mu_k

mean_trig = out_mean_k.subs(trig_subs)

# t -> 0 limit (s fixed, say s = 0.5 for concreteness)
lim_t0 = limit(mean_trig.subs(s, Rational(1,2)), t, 0, '+')
print('limit t->0 (s=0.5):', simplify(lim_t0))

# t -> 1 limit
lim_t1 = limit(mean_trig.subs(s, Rational(1,2)), t, 1, '-')
print('limit t->1 (s=0.5):', simplify(lim_t1))

## Section 6 — Numerical Verification (PyTorch)

3-way comparison:
- **(a)** General formula with `alpha = 1 - sigma` → must match existing `gmflow_posterior_mean_jit`
- **(b)** General formula with trig schedule → must match brute-force (adaptive grid)
- **(c)** Brute-force: evaluate GM log-pdf at grid of x0 values, weight by likelihood, take weighted mean

In [ ]:
import torch
import math

torch.manual_seed(42)
B, K, C = 2, 4, 8  # batch, components, channels

# Shape convention (no spatial dims):
#   schedule scalars : (B, 1)   — broadcasts correctly against (B, C)
#   observations     : (B, C)
#   gm_means         : (B, K, C)
#   gm_vars          : (B, 1, 1) — broadcasts over K and C
#   gm_logweights    : (B, K, 1)

def posterior_general(alpha_t, sigma_t, alpha_s, sigma_s, x_t, x_s,
                      gm_means, gm_vars, gm_logweights, eps=1e-6):
    """Schedule-agnostic posterior mean. Returns (B, C)."""
    sigma_t = sigma_t.clamp(min=eps)
    sigma_s = sigma_s.clamp(min=eps)

    zeta = (alpha_t / sigma_t).square() - (alpha_s / sigma_s).square()  # (B, 1)
    nu   = alpha_t * x_t / sigma_t**2 - alpha_s * x_s / sigma_s**2     # (B, C)

    nu   = nu.unsqueeze(1)    # (B, 1, C) — broadcast over K
    zeta = zeta.unsqueeze(1)  # (B, 1, 1) — broadcast over K and C
    denom = (gm_vars * zeta + 1).clamp(min=eps)                         # (B, 1, 1)

    out_means  = (gm_vars * nu + gm_means) / denom                      # (B, K, C)
    logw_delta = (gm_means * (nu - 0.5 * zeta * gm_means)).sum(
        dim=-1, keepdim=True) / denom                                   # (B, K, 1)
    weights    = (gm_logweights + logw_delta).softmax(dim=1)            # (B, K, 1)
    return (out_means * weights).sum(dim=1)                             # (B, C)


def gmflow_posterior_mean_jit_ref(sigma_s, sigma_t, x_s, x_t,
                                   gm_means, gm_vars, gm_logweights, eps=1e-6):
    """Reference: exact logic from gmflow_posterior_mean_jit (alpha = 1 - sigma)."""
    sigma_s = sigma_s.clamp(min=eps)
    sigma_t = sigma_t.clamp(min=eps)
    alpha_s = 1 - sigma_s
    alpha_t = 1 - sigma_t
    aos_s   = alpha_s / sigma_s
    aos_t   = alpha_t / sigma_t
    zeta = aos_t.square() - aos_s.square()
    nu   = aos_t * x_t / sigma_t - aos_s * x_s / sigma_s
    nu   = nu.unsqueeze(1)
    zeta = zeta.unsqueeze(1)
    denom = (gm_vars * zeta + 1).clamp(min=eps)
    out_means  = (gm_vars * nu + gm_means) / denom
    logw_delta = (gm_means * (nu - 0.5 * zeta * gm_means)).sum(
        dim=-1, keepdim=True) / denom
    weights    = (gm_logweights + logw_delta).softmax(dim=1)
    return (out_means * weights).sum(dim=1)


# --- Test data ---
gm_means      = torch.randn(B, K, C)
gm_logweights = torch.randn(B, K, 1)
gm_vars       = torch.rand(B, 1, 1).abs() + 0.1   # (B, 1, 1)

sigma_s_val = torch.rand(B, 1) * 0.4 + 0.4        # (B, 1), source 0.4–0.8
sigma_t_val = (sigma_s_val * torch.rand(B, 1) * 0.6).clamp(min=1e-3)  # (B, 1), cleaner

alpha_s_lin = 1 - sigma_s_val
alpha_t_lin = 1 - sigma_t_val

x_s = torch.randn(B, C)
x_t = torch.randn(B, C)

# (a) General formula with linear schedule
out_a = posterior_general(
    alpha_t_lin, sigma_t_val, alpha_s_lin, sigma_s_val,
    x_t, x_s, gm_means, gm_vars, gm_logweights)

# (b) Reference (existing gmflow_posterior_mean_jit logic)
out_b = gmflow_posterior_mean_jit_ref(
    sigma_s_val, sigma_t_val, x_s, x_t,
    gm_means, gm_vars, gm_logweights)

diff_ab = (out_a - out_b).abs().max().item()
print(f'(a) vs (b) max |diff| [should be ~0]: {diff_ab:.2e}')
assert diff_ab < 1e-5, f'Linear schedule mismatch: {diff_ab}'
print('PASS: general formula (alpha=1-sigma) == existing code')

In [ ]:
# (c) Trig (VP) schedule: general formula vs exact 1D brute-force
#
# Key insight: the formula computes p(x0 | x_t) by importance-reweighting
# the GM prior p(x0 | x_s) with the ratio p(x_t|x0) / p(x_s|x0):
#
#   log p(x0 | x_t) ∝ log p(x_t|x0) - log p(x_s|x0) + log p(x0|x_s)
#
# This is NOT p(x0 | x_s, x_t) — it replaces x_s information with x_t information.
# The brute-force must reflect this same ratio.

t_val = torch.rand(B, 1) * 0.4 + 0.3                          # (B, 1), t in (0.3, 0.7)
s_val = (t_val + torch.rand(B, 1) * 0.2 + 0.1).clamp(max=0.95)  # s > t (noisier)

alpha_t_trig = torch.cos(math.pi * t_val / 2)   # (B, 1)
sigma_t_trig = torch.sin(math.pi * t_val / 2)
alpha_s_trig = torch.cos(math.pi * s_val / 2)
sigma_s_trig = torch.sin(math.pi * s_val / 2)

x0_true = torch.randn(B, C)
x_t_vp  = alpha_t_trig * x0_true + sigma_t_trig * torch.randn(B, C)  # (B, C)
x_s_vp  = alpha_s_trig * x0_true + sigma_s_trig * torch.randn(B, C)

# Analytic general formula
out_trig = posterior_general(
    alpha_t_trig, sigma_t_trig, alpha_s_trig, sigma_s_trig,
    x_t_vp, x_s_vp, gm_means, gm_vars, gm_logweights)


def brute_force_1d(alpha_t, sigma_t, alpha_s, sigma_s,
                   mu_k_vals, var_k_val, logw_k_vals,
                   x_t_val, x_s_val, n_grid=20000, std_range=12):
    """
    Exact 1D brute force for a single channel and single sample.
    Computes the ratio-reweighted posterior:
      log post ∝ log p(x_t|x0) - log p(x_s|x0) + log p(x0|x_s)
    """
    K  = len(mu_k_vals)
    at, st = float(alpha_t), float(sigma_t)
    as_, ss = float(alpha_s), float(sigma_s)
    xt, xs = float(x_t_val), float(x_s_val)
    vk = float(var_k_val)

    # Centre grid on analytic posterior mean of first component (for stability)
    nu_sc   = at*xt/st**2 - as_*xs/ss**2
    zeta_sc = at**2/st**2 - as_**2/ss**2
    center  = (float(mu_k_vals[0]) + vk*nu_sc) / max(vk*zeta_sc + 1, 1e-8)
    half    = std_range * (vk**0.5 + st / max(at, 1e-8))
    grid    = torch.linspace(center - half, center + half, n_grid)

    # log p(x0|x_s) — GM mixture
    log_gm = torch.stack([
        float(logw_k_vals[k]) - 0.5*(grid - float(mu_k_vals[k]))**2 / vk
        for k in range(K)
    ]).logsumexp(dim=0)

    # log p(x_t|x0) - log p(x_s|x0)  — the importance ratio
    log_ratio = (-0.5*(xt - at*grid)**2/st**2) - (-0.5*(xs - as_*grid)**2/ss**2)

    log_post = log_gm + log_ratio
    log_post = log_post - log_post.logsumexp(dim=0)
    return float((log_post.exp() * grid).sum())


# Run 1D exact test: single sample (b=0), single channel (c=0)
b, c = 0, 0
mean_analytic = float(out_trig[b, c])
mean_brute = brute_force_1d(
    float(alpha_t_trig[b, 0]), float(sigma_t_trig[b, 0]),
    float(alpha_s_trig[b, 0]), float(sigma_s_trig[b, 0]),
    gm_means[b, :, c], gm_vars[b, 0, 0], gm_logweights[b, :, 0],
    x_t_vp[b, c], x_s_vp[b, c])

diff_1d = abs(mean_analytic - mean_brute)
print(f'[Trig 1D] analytic={mean_analytic:.6f}  brute={mean_brute:.6f}  |diff|={diff_1d:.2e}')
assert diff_1d < 0.01, f'Trig mismatch: {diff_1d}'
print('PASS: trig posterior formula matches exact 1D brute-force (K=4)')

## Section 7 — Analytic zeta Bound + Float32 Edge Stability

Under VP/trig: $\zeta = \cot^2(\pi t/2) - \cot^2(\pi s/2)$  
Near $t \to 0$: $\cot(\pi t/2) \approx 2/(\pi t)$, so $\zeta \approx 4/(\pi^2 t^2)$.  

For float32 safety we need `var_k * zeta` to not overflow.  
Typical `var_k` scale: 0.01–1.0. Float32 max ≈ 3.4e38.  
**Safe bound:** `ZETA_MAX = float32_max / max_expected_var_k`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Analytic bound derivation ---
float32_max   = torch.finfo(torch.float32).max
max_var_k     = 10.0   # conservative upper bound on gm_vars in practice
ZETA_MAX_safe = float32_max / max_var_k

# Minimum t at which float32 is safe WITHOUT clamping
# zeta = 4 / (pi^2 * t^2) < ZETA_MAX  =>  t > 2 / (pi * sqrt(ZETA_MAX))
t_min_safe = 2.0 / (math.pi * math.sqrt(ZETA_MAX_safe))

print(f'float32 max:        {float32_max:.3e}')
print(f'max_var_k assumed:  {max_var_k}')
print(f'ZETA_MAX_safe:      {ZETA_MAX_safe:.3e}')
print(f'Minimum safe t (no clamp): {t_min_safe:.3e}')
print(f'  => below t ≈ {t_min_safe:.1e}, zeta * var_k overflows float32')
print(f'  => clamp zeta at {ZETA_MAX_safe:.3e} for safety')

# --- Sweep ---
t_vals = np.linspace(1e-4, 1.0 - 1e-4, 2000)
s_fixed = 0.8  # fixed source time

zeta_trig_np = (np.cos(np.pi * t_vals / 2) / np.sin(np.pi * t_vals / 2))**2 \
             - (np.cos(np.pi * s_fixed / 2) / np.sin(np.pi * s_fixed / 2))**2
zeta_lin_np  = ((1 - t_vals) / t_vals)**2 - ((1 - s_fixed) / s_fixed)**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(t_vals, np.abs(zeta_trig_np), label='VP/trig zeta', color='steelblue')
axes[0].semilogy(t_vals, np.abs(zeta_lin_np),  label='VE/linear zeta', color='darkorange')
axes[0].axhline(ZETA_MAX_safe, color='red', linestyle='--', label='ZETA_MAX (float32 safe)')
axes[0].axvline(t_min_safe, color='red', linestyle=':', alpha=0.7, label=f't_min = {t_min_safe:.1e}')
axes[0].set_xlabel('t'); axes[0].set_ylabel('|zeta|')
axes[0].set_title('zeta: VP vs VE (s=0.8 fixed)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# SNR curves
snr_trig = (np.cos(np.pi * t_vals / 2) / np.sin(np.pi * t_vals / 2))**2
snr_lin  = ((1 - t_vals) / t_vals)**2
axes[1].semilogy(t_vals, snr_trig, label='VP SNR = cot²(πt/2)', color='steelblue')
axes[1].semilogy(t_vals, snr_lin,  label='VE SNR = ((1-t)/t)²', color='darkorange')
axes[1].set_xlabel('t'); axes[1].set_ylabel('SNR')
axes[1].set_title('SNR: VP vs VE')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('section7_stability.png', dpi=120)
plt.show()
print('Saved: section7_stability.png')

In [ ]:
# Float32 overflow test: confirm clamp prevents overflow
t_tiny = torch.tensor([[[1e-5]]])   # near t=0, VP danger zone
s_mid  = torch.tensor([[[0.5]]])
alpha_t_tiny = torch.cos(math.pi * t_tiny / 2)
sigma_t_tiny = torch.sin(math.pi * t_tiny / 2)
alpha_s_mid  = torch.cos(math.pi * s_mid  / 2)
sigma_s_mid  = torch.sin(math.pi * s_mid  / 2)

zeta_raw = (alpha_t_tiny / sigma_t_tiny.clamp(min=1e-6)).square() \
         - (alpha_s_mid  / sigma_s_mid ).square()

print(f'Raw zeta at t=1e-5:    {zeta_raw.item():.3e}')
print(f'Is inf?                {torch.isinf(zeta_raw).item()}')

zeta_clamped = zeta_raw.clamp(max=ZETA_MAX_safe)
var_test     = torch.tensor([[[max_var_k]]])
denom_test   = var_test * zeta_clamped + 1

print(f'After clamp:           {zeta_clamped.item():.3e}')
print(f'denom (var * zeta + 1): {denom_test.item():.3e}')
print(f'Is inf?                {torch.isinf(denom_test).item()}')
print('PASS' if not torch.isinf(denom_test) else 'FAIL')

## Section 8 — LaTeX Export

In [ ]:
from sympy import *
alpha_t, sigma_t = symbols(r'\alpha_t \sigma_t', positive=True)
alpha_s, sigma_s = symbols(r'\alpha_s \sigma_s', positive=True)
x_t, x_s, mu_k, var_k = symbols(r'x_t x_s \mu_k \mathrm{var}_k', real=True)

zeta_expr   = (alpha_t/sigma_t)**2 - (alpha_s/sigma_s)**2
nu_expr     = alpha_t*x_t/sigma_t**2 - alpha_s*x_s/sigma_s**2
denom_expr  = var_k * zeta_expr + 1
mean_expr   = (var_k * nu_expr + mu_k) / denom_expr
logw_expr   = mu_k * (nu_expr - Rational(1,2)*zeta_expr*mu_k) / denom_expr

print('% === General-form GMFlow Posterior (schedule-agnostic) ===')
print(f'\\zeta   &= {latex(zeta_expr)}')
print(f'\\nu     &= {latex(nu_expr)}')
print(f'd_k     &= {latex(denom_expr)}')
print(f'\\bar{{\\mu}}_k &= {latex(mean_expr)}')
print(f'\\Delta \\log w_k &= {latex(logw_expr)}')